# Notebook 2 · Final pipeline, visualization (bonus) and summary

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"
DEBUG = False

os.environ["VISDET_DEBUG"] = "1" if DEBUG else "0"
if os.path.isdir("/kaggle/working"):
    if not os.path.isdir("/kaggle/working/repo"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "/kaggle/working/repo"], check=True)
    CODE = "/kaggle/working/repo/code"
else:
    CODE = os.environ.get("VISDET_CODE") or os.path.abspath("..")
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
if os.environ.get("VISDET_SKIP_PIP") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

## 06 · Final pipeline

In [ ]:
USE_P2 = False
USE_NWD = False
FINAL_BUDGET_H = 3.0

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from visdet.common import FINAL_STEP, VAL_STEP, data_dir, find_comp_root, is_debug, setup_wandb_from_kaggle_secret, work_dir
from visdet import evaluate as ev, infer, prep
from visdet.results import collect, export_results
from visdet.runner import run_jobs, ul_job
from visdet.ul_train import load_trained

setup_wandb_from_kaggle_secret()
COMP, DATA, RUNS = find_comp_root(), data_dir(), work_dir() / "runs"
FIG = work_dir() / "figures"; FIG.mkdir(exist_ok=True)


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
tr, va, yaml_path = prep.prepare_data(COMP, DATA, FINAL_STEP, VAL_STEP)
GT = ev.build_gt(va)
print(f"train {len(tr)} frames (every {FINAL_STEP}rd) | val {len(va)} frames (unchanged)")

### Part 1 — training the final candidates (in parallel, one per GPU)

In [ ]:
extra = dict(p2=USE_P2, nwd=USE_NWD)
FINAL = {
    "final_A_yolo11s": dict(family="yolo", model="yolo11s", batch=6 if USE_P2 else 8, **extra),
    "final_B_yolo11m": dict(family="yolo", model="yolo11m", batch=4 if USE_P2 else 6, **extra),
}
codes = run_jobs([ul_job(e, yaml_path, RUNS, FINAL_BUDGET_H, cfg) for e, cfg in FINAL.items()], RUNS / "logs",
                 report_every_s=1800)
print(codes)

In [ ]:
res = collect(["final", "yolo", "rtdetr", "retina", "frcnn"], extra=[str(RUNS)])
res.set_index("exp")[["mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_img", "epochs"]].round(4)

### Part 2 — inference strategies (same weights, val frames)

In [ ]:
fin = res[res.exp.str.startswith("final")].sort_values("mAP", ascending=False)
BEST = fin["exp"].iloc[0]
OTHER = fin["exp"].iloc[1] if len(fin) > 1 else None
model = load_trained(RUNS / BEST / "best.pt")
paths, stems = va["src_img"].tolist(), va["stem"].tolist()
STRATS = {
    "S0 full 1280": dict(imgsz=1280),
    "S1 full 1536": dict(imgsz=1536),
    "S2 full 1920": dict(imgsz=1920),
    "S3 full 1280 + 2x2 tiles (NMS)": dict(imgsz=1280, tile_grid=(2, 2), tile_imgsz=1280),
    "S4 full 1280 + 2x2 tiles (WBF)": dict(imgsz=1280, tile_grid=(2, 2), tile_imgsz=1280, merge="wbf"),
    "S5 TTA flip+scale 1280": dict(imgsz=1280, augment=True),
}
if is_debug():
    STRATS = {k: {**v, "imgsz": 320, "tile_imgsz": 320} for k, v in list(STRATS.items())[:4]}
val_preds, rows = {}, []
for name, s in STRATS.items():
    val_preds[name] = infer.predict_many(model, paths, stems, s, log_every=0)
    m = ev.coco_eval(GT, val_preds[name], per_class=False)
    rows.append(dict(strategy=name, mAP=m["mAP"], AP50=m["AP50"], AP_small=m["AP_small"], AP_medium=m["AP_medium"]))
    print(rows[-1])
strat = pd.DataFrame(rows).set_index("strategy")
strat.round(4)

#### Ensemble of the two final models (WBF), each with the best single-model strategy

In [ ]:
BEST_S = strat["mAP"].idxmax()
sizes = {r.stem: (int(r.orig_w), int(r.orig_h)) for r in va.itertuples()}
if OTHER:
    model2 = load_trained(RUNS / OTHER / "best.pt")
    p2 = infer.predict_many(model2, paths, stems, STRATS[BEST_S], log_every=0)
    ens = infer.ensemble([val_preds[BEST_S], p2], sizes, weights=[1.0, 1.0])
    m = ev.coco_eval(GT, ens, per_class=False)
    strat.loc[f"S6 WBF({BEST}+{OTHER}) @ {BEST_S[:2]}"] = [m["mAP"], m["AP50"], m["AP_small"], m["AP_medium"]]
display(strat.round(4))
fig, ax = plt.subplots(figsize=(8, 3.5))
strat[["mAP", "AP_small"]].plot.barh(ax=ax); ax.set_title(f"Inference strategies ({BEST})"); ax.invert_yaxis()
save(fig, "final_strategies")

### Test predictions with the selected pipeline

In [ ]:
CHOICE = strat["mAP"].idxmax()
print("selected:", CHOICE)
test = pd.read_csv(COMP / "test_images.csv")
tpaths, tids = [str(COMP / f) for f in test["file_name"]], test["image_id"].tolist()
SUBS = work_dir() / "submissions"; SUBS.mkdir(exist_ok=True)

safe = infer.predict_many(model, tpaths, tids, STRATS["S0 full 1280"] if "S0 full 1280" in STRATS else list(STRATS.values())[0])
infer.check_submission(infer.write_submission(safe, test, SUBS / f"sub_{BEST}_S0.csv"), test)

if CHOICE.startswith("S6"):
    a = infer.predict_many(model, tpaths, tids, STRATS[BEST_S])
    b = infer.predict_many(model2, tpaths, tids, STRATS[BEST_S])
    tsizes = {r.image_id: (int(r.width), int(r.height)) for r in test.itertuples()}
    final = infer.ensemble([a, b], tsizes, weights=[1.0, 1.0])
elif CHOICE.startswith("S0"):
    final = safe
else:
    final = infer.predict_many(model, tpaths, tids, STRATS[CHOICE])
sub = infer.write_submission(final, test, SUBS / "submission_final.csv")
infer.check_submission(sub, test)

## 07 · Bonus: what the models learn and where they fail

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from visdet.common import (CLASSES, TRAIN_STEP, VAL_STEP, data_dir, find_comp_root, find_file, is_debug,
                           setup_wandb_from_kaggle_secret, work_dir)
from visdet.configs import TV_COMMON, TV_EXPS, UL_COMMON
from visdet import evaluate as ev, prep, viz, wb
from visdet.results import collect

setup_wandb_from_kaggle_secret()
COMP, DATA = find_comp_root(), data_dir()
FIG = work_dir() / "figures"; FIG.mkdir(exist_ok=True)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RUN = wb.init("visualization", {}, job_type="analysis")
FIGS = []


def save(fig, name):
    p = FIG / f"{name}.png"; fig.savefig(p, dpi=130, bbox_inches="tight"); FIGS.append(p); plt.show()

In [ ]:
va = prep.build_split(COMP, "val", VAL_STEP, DATA)
GT = ev.build_gt(va); G = ev.gt_per_image(GT)
src = va.set_index("stem")["src_img"]
res = collect(["frcnn", "rtdetr"])
pick = lambda pre, default: (res[res.exp.str.startswith(pre)].sort_values("mAP")["exp"].iloc[-1]
                             if res.exp.str.startswith(pre).any() else default)
FR_EXP, RT_EXP = pick("frcnn", "frcnn_E1_anchors"), pick("rtdetr", "rtdetr_E1_q500")
FR_W = find_file(f"{FR_EXP}/model_final.pth") or find_file("frcnn_*/model_final.pth")
RT_W = find_file(f"{RT_EXP}/best.pt") or find_file("rtdetr_*/best.pt")
print("Faster R-CNN:", FR_W, "\nRT-DETR:", RT_W)

### 1. Grad-CAM on Faster R-CNN

In [ ]:
from visdet.tv_models import build_tv_model
from visdet.tv_train import load_rgb, predict_tv
exp_name = FR_W.parent.name
fr = build_tv_model({**TV_COMMON, **TV_EXPS[exp_name]}).to(DEV)
fr.load_state_dict(torch.load(FR_W, map_location=DEV)); fr.eval()
fr_preds = predict_tv(fr, va["src_img"].tolist(), va["stem"].tolist(), 1, DEV)

In [ ]:
_, per = ev.error_breakdown(GT, fr_preds, score_thr=0.3)
stem = per.sort_values("n_gt", ascending=False)["stem"].iloc[min(2, len(per) - 1)]
gb, gl, _ = G[stem]; p = fr_preds[stem]
ious = ev._iou(p["boxes"], gb)
area = (p["boxes"][:, 2] - p["boxes"][:, 0]) * (p["boxes"][:, 3] - p["boxes"][:, 1])
tp = (ious.max(1, initial=0) >= 0.5) & (p["scores"] >= 0.5) if len(gb) else np.zeros(len(area), bool)
fp = (ious.max(1, initial=0) < 0.1) & (p["scores"] >= 0.3) if len(gb) else p["scores"] >= 0.3
cases = {}
if tp.any():
    cases["small TP"] = dict(idx=int(np.where(tp)[0][np.argmin(area[tp])]))
    cases["large TP"] = dict(idx=int(np.where(tp)[0][np.argmax(area[tp])]))
if fp.any():
    cases["false positive"] = dict(idx=int(np.where(fp)[0][np.argmax(p["scores"][fp])]))
if len(gb):
    hit = ious.max(0) >= 0.5 if len(p["boxes"]) else np.zeros(len(gb), bool)
    garea = (gb[:, 2] - gb[:, 0]) * (gb[:, 3] - gb[:, 1])
    miss = np.where(~hit)[0]
    if len(miss):
        cases["missed GT (nearest det)"] = dict(box=gb[miss[np.argmin(garea[miss])]])
if not cases and len(p["scores"]):
    cases["top detection"] = dict(idx=int(np.argmax(p["scores"])))

cam = viz.FRCNNGradCAM(fr)
img_t = load_rgb(src[stem]).to(DEV); img = viz.rgb(src[stem])
levels = ["P2", "P3", "P4", "P5", "C2", "C3", "C4", "C5"]
fig, axes = plt.subplots(len(cases), 1 + len(levels), figsize=(2.2 * (1 + len(levels)), 2.4 * len(cases)))
axes = np.atleast_2d(axes)
for row, (name, c) in zip(axes, cases.items()):
    det, cams, i = cam(img_t, det_index=c.get("idx"), box=c.get("box"))
    b = c.get("box", det["boxes"][i])
    row[0].imshow(viz.crop_zoom(viz.draw_boxes(img, [b], color=(0, 255, 0)), b, pad=60))
    row[0].set_title(f"{name}\n{CLASSES[det['labels'][i] - 1]} {det['scores'][i]:.2f}", fontsize=8)
    for ax, lv in zip(row[1:], levels):
        ax.imshow(viz.crop_zoom(viz.cam_overlay(img, cams.get(lv, np.zeros(img.shape[:2]))), b, pad=60))
        ax.set_title(lv, fontsize=8)
for ax in axes.ravel():
    ax.axis("off")
save(fig, "viz_gradcam_cases")

#### Grad-CAM statistics over many detections

In [ ]:
rows = []
stems = va["stem"].tolist()[:: max(1, len(va) // 20)]
for s in stems:
    gb, gl, _ = G[s]; p = fr_preds[s]
    if not len(gb) or not len(p["boxes"]):
        continue
    ious = ev._iou(p["boxes"], gb)
    cand = np.where((ious.max(1) >= 0.5) & (p["scores"] >= 0.5))[0][:3]
    t = load_rgb(src[s]).to(DEV)
    for k in cand:
        det, cams, i = cam(t, box=p["boxes"][k])
        x1, y1, x2, y2 = det["boxes"][i].round().astype(int)
        a = (x2 - x1) * (y2 - y1)
        for lv in levels:
            c = cams.get(lv)
            if c is None or c.sum() == 0:
                continue
            rows.append(dict(level=lv, size="small" if a < 32 ** 2 else "medium" if a < 96 ** 2 else "large",
                             inside=float(c[y1:y2, x1:x2].sum() / c.sum())))
cam.remove()
cs = pd.DataFrame(rows)
if len(cs):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    cs.pivot_table(index="level", columns="size", values="inside", aggfunc="mean").reindex(levels).plot.bar(ax=ax)
    ax.set_ylabel("CAM mass inside the box"); ax.set_title("Where the evidence comes from, by stage and object size")
    save(fig, "viz_gradcam_stats")
    display(cs.groupby(["size", "level"]).inside.agg(["mean", "count"]).round(3).unstack(0))

### 2. RoI embeddings (UMAP)

In [ ]:
from visdet.viz import roi_embeddings
E, Y, S, OK = [], [], [], []
for s in va["stem"].tolist()[:: max(1, len(va) // 60)]:
    gb, gl, _ = G[s]
    if not len(gb):
        continue
    k = np.random.default_rng(0).permutation(len(gb))[:80]
    emb, prob = roi_embeddings(fr, load_rgb(src[s]).to(DEV), gb[k])
    E.append(emb); Y.append(gl[k]); S.append(np.sqrt((gb[k, 2] - gb[k, 0]) * (gb[k, 3] - gb[k, 1])))
    OK.append(prob[:, 1:].argmax(1) == gl[k])
E, Y, S, OK = map(np.concatenate, (E, Y, S, OK))
print(E.shape, "box classifier accuracy on GT boxes:", OK.mean().round(3))
try:
    import umap
    Z = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=0).fit_transform(E)
    method = "UMAP"
except Exception as e:
    from sklearn.manifold import TSNE
    Z = TSNE(perplexity=min(30, len(E) - 1), random_state=0).fit_transform(E); method = "t-SNE"

fig, ax = plt.subplots(1, 3, figsize=(18, 5.5))
for c in range(len(CLASSES)):
    m = Y == c
    ax[0].scatter(Z[m, 0], Z[m, 1], s=3, color=np.array(viz.PALETTE[c]) / 255, label=CLASSES[c])
ax[0].legend(markerscale=4, fontsize=7); ax[0].set_title(f"{method} of RoI features — class")
sc = ax[1].scatter(Z[:, 0], Z[:, 1], s=3, c=np.log2(S), cmap="viridis"); plt.colorbar(sc, ax=ax[1], label="log2 √area (px)")
ax[1].set_title("object size")
ax[2].scatter(Z[OK, 0], Z[OK, 1], s=3, c="lightgray", label="correct class")
ax[2].scatter(Z[~OK, 0], Z[~OK, 1], s=4, c="crimson", label="wrong class"); ax[2].legend(markerscale=4); ax[2].set_title("box classifier correct?")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
save(fig, "viz_roi_umap")
acc = pd.DataFrame(dict(cls=[CLASSES[y] for y in Y], ok=OK, small=S < 32)).groupby(["cls", "small"]).ok.mean().unstack()
acc.round(3)

### 3. RT-DETR decoder cross-attention

In [ ]:
from visdet.ul_train import load_trained
from visdet.viz import RTDETRAttention
rt = load_trained(RT_W)
net = rt.model.to(DEV).eval()
IMGSZ = 320 if is_debug() else UL_COMMON["imgsz"]
att = RTDETRAttention()
img = viz.rgb(src[stem])
boxes, scores, labels, recs = att.run(net, img, IMGSZ)
print(len(recs), "deformable-attention calls; sampling tensor:", tuple(recs[0]["loc"].shape))

In [ ]:
def plot_query(ax, q, layer, img, box, name=""):
    loc, w = recs[layer]["loc"][0, q], recs[layer]["w"][0, q]
    H, W = img.shape[:2]
    xy = loc.reshape(-1, 2).numpy() * [W, H]
    ww = w.reshape(-1).numpy()
    lv = np.repeat(np.arange(loc.shape[1])[None, :, None], loc.shape[0], 0).repeat(loc.shape[2], 2).reshape(-1)
    x1, y1, x2, y2 = box; pad = max(60, 1.5 * max(x2 - x1, y2 - y1))
    ax.imshow(img); ax.set_xlim(x1 - pad, x2 + pad); ax.set_ylim(y2 + pad, y1 - pad)
    ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="lime", lw=1.5))
    ax.scatter(xy[:, 0], xy[:, 1], s=120 * ww / ww.max(), c=lv, cmap="cool", alpha=.8, vmin=0, vmax=2)
    ax.set_title(f"{name} · layer {layer + 1}", fontsize=8); ax.axis("off")


area_q = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
conf_q = np.where(scores > 0.3)[0] if (scores > 0.3).any() else np.argsort(-scores)[:10]
sel = {"small object": conf_q[np.argmin(area_q[conf_q])], "large object": conf_q[np.argmax(area_q[conf_q])]}
L = len(recs)
fig, axes = plt.subplots(len(sel), 3, figsize=(12, 4 * len(sel)))
for row, (name, q) in zip(np.atleast_2d(axes), sel.items()):
    for ax, layer in zip(row, [0, L // 2, L - 1]):
        plot_query(ax, q, layer, img, boxes[q], name)
fig.suptitle("Deformable sampling points (size = attention weight, colour = feature level: stride 8 → 32)")
save(fig, "viz_rtdetr_sampling")

#### Attention spread and feature-level use, over many queries

In [ ]:
rows = []
for s in va["stem"].tolist()[:: max(1, len(va) // 15)]:
    im = viz.rgb(src[s]); H, W = im.shape[:2]
    b, sc_, lb, rc = att.run(net, im, IMGSZ)
    top = np.argsort(-sc_)[:60]
    for q in top:
        x1, y1, x2, y2 = b[q]; a = (x2 - x1) * (y2 - y1)
        size = "small" if a < 32 ** 2 else "medium" if a < 96 ** 2 else "large"
        diag = max(np.hypot(x2 - x1, y2 - y1), 1.0); cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
        for li, r in enumerate(rc):
            loc, w = r["loc"][0, q].numpy(), r["w"][0, q].numpy()
            d = np.hypot(loc[..., 0] * W - cx, loc[..., 1] * H - cy) / diag
            lvl_mass = w.sum(axis=(0, 2)) / w.sum()
            rows.append(dict(layer=li + 1, size=size, spread=float((w * d).sum() / w.sum()),
                             **{f"level{j}": float(v) for j, v in enumerate(lvl_mass)}))
att.remove()
st = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
st.pivot_table(index="layer", columns="size", values="spread").plot(marker="o", ax=ax[0])
ax[0].set(ylabel="spread / box diagonal", title="Sampling spread per decoder layer")
lv_cols = [c for c in st.columns if c.startswith("level")]
st.groupby("size")[lv_cols].mean().rename(columns=dict(zip(lv_cols, ["stride 8", "stride 16", "stride 32"]))).plot.bar(
    stacked=True, ax=ax[1]); ax[1].set(ylabel="attention mass", title="Which feature level the queries read"); ax[1].tick_params(axis="x", rotation=0)
save(fig, "viz_rtdetr_stats")
st.groupby(["size", "layer"]).spread.mean().unstack(0).round(3)

## 08 · Summary of all models

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from visdet.common import work_dir
from visdet.results import REPO_RESULTS, collect, find_run

FIG = work_dir() / "figures"; FIG.mkdir(exist_ok=True)
RUNS = work_dir() / "runs"; RUNS.mkdir(exist_ok=True)
LB = {
    "frcnn_E1_anchors": None,
    "retina_E1_anchors": None,
    "yolo_E0_baseline": None,
    "rtdetr_E1_q500": None,
    "final_B_yolo11m": None,
}


def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
res = collect(["frcnn", "retina", "yolo", "rtdetr", "final"], extra=[str(REPO_RESULTS)])
res["family"] = res["exp"].str.split("_").str[0]
res["public_LB"] = res["exp"].map(LB)
cols = ["family", "exp", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_img", "epochs", "public_LB"]
res[cols].style.format(precision=4, na_rep="–").background_gradient(subset=["mAP", "AP_small"], cmap="Greens")

In [ ]:
res[cols].round(4).to_csv(RUNS / "summary_all_runs.csv", index=False)
try:
    print(res[cols].round(4).to_markdown(index=False))
except ImportError:
    print(res[cols].round(4).to_csv(index=False))

### Incremental changes inside each family (Δ mAP vs. the family baseline)

In [ ]:
base = res[res.exp.str.contains("E0")].set_index("family")["mAP"]
res["delta_vs_E0"] = res["mAP"] - res["family"].map(base)
fig, ax = plt.subplots(figsize=(10, 4))
d = res[~res.exp.str.contains("E0") & res.family.isin(base.index)].set_index("exp")["delta_vs_E0"]
ax.barh(d.index, d.values, color=np.where(d.values >= 0, "tab:green", "tab:red")); ax.axvline(0, c="k", lw=.8)
ax.set_xlabel("Δ mAP vs family baseline (E0)"); ax.invert_yaxis(); ax.set_title("What each experiment adds")
save(fig, "summary_deltas")

### Best run per family: accuracy by size and speed

In [ ]:
best = res.sort_values("mAP").groupby("family").tail(1).set_index("exp")
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
best[["mAP", "AP_small", "AP_medium", "AP_large"]].plot.bar(ax=ax[0]); ax[0].set_title("Best run per family")
ax[0].tick_params(axis="x", rotation=20)
for e, r in best.iterrows():
    ax[1].scatter(r.ms_per_img, r.mAP, s=60); ax[1].annotate(e, (r.ms_per_img, r.mAP), fontsize=8)
ax[1].set(xlabel="ms / image (T4, batch 1)", ylabel="val mAP", title="Speed vs accuracy")
save(fig, "summary_best")

### Validation curves of all Ultralytics runs (same GPU budget)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for e in res.exp:
    run = find_run(e, extra=[str(REPO_RESULTS)])
    if run is not None and (run / "results.csv").exists():
        h = pd.read_csv(run / "results.csv"); h.columns = [c.strip() for c in h.columns]
        if "time" in h and not e.startswith("final"):
            ax.plot(h["time"] / 60, h["metrics/mAP50-95(B)"], "-" if "rtdetr" in e else "--", label=e)
ax.set(xlabel="training minutes", ylabel="mAP50-95 (Ultralytics val)"); ax.legend(fontsize=7)
save(fig, "summary_ul_curves")

In [ ]:
for e in ["final_B_yolo11m", "final_A_yolo11s"]:
    run = find_run(e, extra=[str(REPO_RESULTS)])
    if run is not None and (run / "strategies.csv").exists():
        print("inference strategies of", e); display(pd.read_csv(run / "strategies.csv").round(4))

In [ ]:
from visdet.results import export_results
export_results(work_dir() / "runs", work_dir() / "results_nb2.zip", figures_dir=FIG)